In [8]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df_users = pd.read_parquet("user1.parquet")
df_rest = pd.read_json("business1.json")
df_review = pd.read_parquet("review1.parquet")

In [ ]:
df_rest_b = df_rest.drop(columns=['name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'is_open', 'HairSpecializesIn', 'AcceptsInsurance'])



In [ ]:
df_users_b = df_users.drop(columns=['compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos', 'friends', 'name', 'fans', 'useful', 'funny',
       'cool', 'elite'])

In [ ]:
df_review_b = df_review.drop(columns=['useful', 'funny',
       'cool', 'date', 'text'])



---

In [ ]:
elt = {}
for val in restaurants_df["BusinessParking"]:
    for i in val:
        

{'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}
None
{'garage': False, 'street': False, 'validated': False, 'lot': False, 'valet': False}
{'garage': False, 'street': False, 'validated': False, 'lot': False, 'valet': False}
{'garage': None, 'street': False, 'validated': None, 'lot': True, 'valet': False}
{u'valet': False, u'garage': None, u'street': True, u'lot': False, u'validated': None}
{'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}
None
{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}
None
{'garage': False, 'street': True, 'validated': False, 'lot': True, 'valet': False}
{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}
{'garage': False, 'street': True, 'validated': False, 'lot': True, 'valet': False}
{u'valet': False, u'garage': False, u'street': False, u'lot': True, u'validated': False}
{'garage': False, 'street': False, 'validated': False, 

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import  StandardScaler
from collections import Counter
import warnings
warnings.filterwarnings('ignore')


# E1:

# Charger les datasets
users_df = pd.read_parquet('user1.parquet')
reviews_df = pd.read_parquet('review1.parquet')
restaurants_df = pd.read_json("business1.json")

reviews_df = reviews_df.drop(columns=['useful', 'funny',
       'cool', 'date', 'text'])

restaurants_df = restaurants_df.drop(columns=['name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'is_open', 'HairSpecializesIn', 'AcceptsInsurance'])

users_df = users_df.drop(columns=['compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos', 'friends', 'name', 'fans', 'useful', 'funny',
       'cool', 'elite'])

# E2: Subset

# Sélectionner les users les plus actifs
top_users = users_df.nlargest(10000, 'review_count')['user_id'].values

# Filtrer les reviews
subset_reviews = reviews_df[reviews_df['user_id'].isin(top_users)].copy()
top_restaurants = subset_reviews['business_id'].value_counts().head(5000).index.values
subset_reviews = subset_reviews[subset_reviews['business_id'].isin(top_restaurants)].copy()

# Filtrer les dataframes
subset_users = users_df[users_df['user_id'].isin(subset_reviews['user_id'].unique())].copy()
subset_restaurants = restaurants_df[restaurants_df['business_id'].isin(top_restaurants)].copy()

# E3:preprocessing

subset_users['review_count_log'] = np.log1p(subset_users['review_count'])
subset_users['average_stars_norm'] = subset_users['average_stars'] / 5.0

# Ancienneté
subset_users['yelping_since'] = pd.to_datetime(subset_users['yelping_since'])
subset_users['years_active'] = (pd.Timestamp.now() - subset_users['yelping_since']).dt.days / 365.25
subset_users['years_active_norm'] = subset_users['years_active'] / subset_users['years_active'].max()

user_features_cols = ['review_count_log', 'average_stars_norm', 'years_active_norm']

# Features numériques de base
subset_restaurants['stars_norm'] = subset_restaurants['stars'] / 5.0
subset_restaurants['review_count_log'] = np.log1p(subset_restaurants['review_count'])

# Price Range
subset_restaurants['RestaurantsPriceRange2'] = pd.to_numeric(subset_restaurants['RestaurantsPriceRange2'], errors='coerce').fillna(2.0) 
subset_restaurants['price_norm'] = subset_restaurants['RestaurantsPriceRange2'] / 4.0

# Categories
all_categories = []
for cats in subset_restaurants['categories'].dropna():
    if isinstance(cats, str):
        all_categories.extend([c.strip() for c in cats.split(',')])

top_20_categories = [cat for cat, _ in Counter(all_categories).most_common(20)]

for cat in top_20_categories:
    subset_restaurants[f'cat_{cat}'] = subset_restaurants['categories'].apply(lambda x: 1.0 if isinstance(x, str) and cat in x else 0.0)

# Attributs booléens
boolean_attrs = [
    'RestaurantsDelivery', 'OutdoorSeating', 'BusinessAcceptsCreditCards',
    'BikeParking', 'RestaurantsTakeOut', 'GoodForKids', 'Caters',
    'RestaurantsReservations', 'RestaurantsTableService', 'RestaurantsGoodForGroups',
    'WheelchairAccessible', 'HasTV', 'HappyHour', 'DriveThru',
    'BusinessAcceptsBitcoin', 'DogsAllowed', 'Open24Hours'
]

def parse_boolean(val):
    if pd.isna(val) or val is None or val == 'None':
        return 0.0
    if isinstance(val, bool):
        return 1.0 if val else 0.0
    if isinstance(val, str):
        val_lower = val.lower()
        if val_lower in ['true', '1', 'yes']:
            return 1.0
        elif val_lower in ['false', '0', 'no']:
            return 0.0
    return 0.0

for attr in boolean_attrs:
    if attr in subset_restaurants.columns:
        subset_restaurants[f'{attr}_bool'] = subset_restaurants[attr].apply(parse_boolean)

# Attributs catégoriels
categorical_attrs = {
    'WiFi': ['free', 'no', 'paid'],
    'Alcohol': ['full_bar', 'beer_and_wine', 'none'],
    'NoiseLevel': ['quiet', 'average', 'loud', 'very_loud'],
    'RestaurantsAttire': ['casual', 'dressy', 'formal'],
    'Smoking': ['no', 'yes', 'outdoor'],
    "BusinessParking": []
}

for attr, values in categorical_attrs.items():
    if attr in subset_restaurants.columns:
        for val in values:
            subset_restaurants[f'{attr}_{val}'] = subset_restaurants[attr].apply(lambda x: 1.0 if isinstance(x, str) and val in x.lower() and x!="None" else 0.0)

# Compiler toutes les features restaurants
restaurant_features_cols = (
    ['stars_norm', 'review_count_log', 'price_norm'] +
    [f'cat_{cat}' for cat in top_20_categories] +
    [f'{attr}_bool' for attr in boolean_attrs if attr in subset_restaurants.columns] +
    [f'{attr}_{val}' for attr, values in categorical_attrs.items() 
     for val in values if attr in subset_restaurants.columns]
)

subset_restaurants_fillna = subset_restaurants[restaurant_features_cols].fillna(0)
subset_users_fillna = subset_users[user_features_cols].fillna(0)

In [13]:
# E4: Création du Graphe

# Créer les mappings
user_to_idx = {uid: i for i, uid in enumerate(subset_users['user_id'])}
num_users = len(user_to_idx)
restaurant_to_idx = {rid: i+ num_users for i, rid in enumerate(subset_restaurants['business_id'])}


num_restaurants = len(restaurant_to_idx)


# Normaliser les features
scaler_user = StandardScaler()
scaler_restaurant = StandardScaler()

user_features = scaler_user.fit_transform(subset_users_fillna)
restaurant_features = scaler_restaurant.fit_transform(subset_restaurants_fillna)

# Padding pour avoir la même dimension
max_dim = max(user_features.shape[1], restaurant_features.shape[1])

user_features_padded = np.pad(
    user_features,
    ((0, 0), (0, max_dim - user_features.shape[1])),
    mode='constant'
)
restaurant_features_padded = np.pad(
    restaurant_features,
    ((0, 0), (0, max_dim - restaurant_features.shape[1])),
    mode='constant'
)

# Combiner
all_features = np.vstack([user_features_padded, restaurant_features_padded])
x = torch.FloatTensor(all_features)

edge_list = []
edge_labels = []

# Likes 
likes = subset_reviews[subset_reviews['stars'] >= 3]
for _, row in likes.iterrows():
    if row['user_id'] in user_to_idx and row['business_id'] in restaurant_to_idx:
        user_idx = user_to_idx[row['user_id']]
        restaurant_idx = restaurant_to_idx[row['business_id']]
        
        edge_list.append([user_idx, restaurant_idx])
        edge_list.append([restaurant_idx, user_idx])
        edge_labels.extend([1.0, 1.0])

# Dislikes
dislikes = subset_reviews[subset_reviews['stars'] <= 2]
for _, row in dislikes.iterrows():
    if row['user_id'] in user_to_idx and row['business_id'] in restaurant_to_idx:
        user_idx = user_to_idx[row['user_id']]
        restaurant_idx = restaurant_to_idx[row['business_id']]
        
        edge_list.append([user_idx, restaurant_idx])
        edge_list.append([restaurant_idx, user_idx])
        edge_labels.extend([0.0, 0.0])

edge_index = torch.LongTensor(edge_list).t().contiguous()
edge_label = torch.FloatTensor(edge_labels)


print(f"Edges: {edge_index.shape[1]:,}")
print(f"Likes: {(edge_label == 1.0).sum().item():,}")
print(f"Dislikes: {(edge_label == 0.0).sum().item():,}")

graph_data = Data(x=x, edge_index=edge_index, edge_label=edge_label)


Edges: 337,124
Likes: 310,604
Dislikes: 26,520


In [14]:
# E5: GNN

class RestaurantGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.predictor = nn.Sequential(
            nn.Linear(out_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, 1)
            
        )
        
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x
    
    def decode(self, z, edge_label_index):
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        return self.predictor(torch.cat([src, dst], dim=1)).squeeze()
    
    def forward(self, x, edge_index, edge_label_index):
        z = self.encode(x, edge_index)
        return self.decode(z, edge_label_index)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RestaurantGNN(
    in_channels=max_dim,
    hidden_channels=128,
    out_channels=64
).to(device)



In [15]:
# E6: Training

optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.BCEWithLogitsLoss()

data = graph_data.to(device)

num_epochs = 50

model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Negative sampling
    neg_edge_index = negative_sampling(
        edge_index=data.edge_index,
        num_nodes=data.num_nodes,
        num_neg_samples=data.edge_index.size(1) // 2
    )
    
    # Combiner edges positifs et négatifs
    edge_label_index = torch.cat([data.edge_index, neg_edge_index], dim=1)
    edge_label = torch.cat([
        data.edge_label,
        torch.zeros(neg_edge_index.size(1))
    ], dim=0).to(device)
    
    # Forward pass
    out = model(data.x, data.edge_index, edge_label_index)
    loss = criterion(out, edge_label)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Metrics
    if epoch % 5 == 0:
        with torch.no_grad():
            pred = out.cpu().numpy()
            true = edge_label.cpu().numpy()
            auc = roc_auc_score(true, pred) if len(np.unique(true)) > 1 else 0.5
            acc = ((pred > 0.5) == true).mean()
            
            print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | AUC: {auc:.4f} | Acc: {acc:.4f}")

# E7: recommandation

def get_top_k_recommendations(user_id, k=20, verbose=True):
    """
    Obtient les top K recommandations pour un user
    
    Args:
        user_id: ID du user
        k: Nombre de recommandations
        verbose: Afficher les détails
    
    Returns:
        List of (restaurant_id, score, restaurant_info)
    """
    if user_id not in user_to_idx:
        print(f"User {user_id} non trouvé dans le subset")
        return []
    
    model.eval()
    with torch.no_grad():
        # Encoder le graphe
        z = model.encode(data.x, data.edge_index)
        
        user_idx = user_to_idx[user_id]
        
        # Calculer les scores pour tous les restaurants
        scores = []
        for restaurant_id, restaurant_idx in restaurant_to_idx.items():
            restaurant_global_idx = restaurant_idx
            
            edge_label_index = torch.tensor(
                [[user_idx], [restaurant_global_idx]], 
                device=device
            )
            score = model.decode(z, edge_label_index).item()
            scores.append((restaurant_id, score))
        
        # Trier par score décroissant
        scores.sort(key=lambda x: x[1], reverse=True)
        top_k = scores[:k]
        
        # Enrichir avec les infos restaurants
        recommendations = []
        for restaurant_id, score in top_k:
            restaurant_info = subset_restaurants[subset_restaurants['business_id'] == restaurant_id].iloc[0].to_dict()
            recommendations.append((restaurant_id, score, restaurant_info))
        
        if verbose:
            print(f"Top {k} recommandations pour user {user_id}:")
            for i, (rid, score, info) in enumerate(recommendations, 1):
                print(f"{i}. Score: {score:.4f} | Stars: {info['stars']:.1f} | {info.get('categories', 'N/A')[:50]}")
        
        return recommendations

# Test avec un user aléatoire
test_user = subset_users.iloc[0]['user_id']
recs = get_top_k_recommendations(test_user, k=20)


Epoch 000 | Loss: 0.6930 | AUC: 0.4653 | Acc: 0.3858
Epoch 005 | Loss: 0.6458 | AUC: 0.6853 | Acc: 0.6585
Epoch 010 | Loss: 0.5408 | AUC: 0.7867 | Acc: 0.7408
Epoch 015 | Loss: 0.4635 | AUC: 0.8032 | Acc: 0.7861
Epoch 020 | Loss: 0.4286 | AUC: 0.8475 | Acc: 0.7971
Epoch 025 | Loss: 0.4585 | AUC: 0.7967 | Acc: 0.8193
Epoch 030 | Loss: 0.3945 | AUC: 0.8677 | Acc: 0.8478
Epoch 035 | Loss: 0.3571 | AUC: 0.8846 | Acc: 0.8528
Epoch 040 | Loss: 0.3463 | AUC: 0.8949 | Acc: 0.8506
Epoch 045 | Loss: 0.3409 | AUC: 0.8982 | Acc: 0.8517
Top 20 recommandations pour user qVc8ODYU5SZjKXVBgXdI7w:
1. Score: 2.9600 | Stars: 4.5 | Candy Stores, Shopping, Department Stores, Fast Fo
2. Score: 2.6908 | Stars: 4.0 | Live/Raw Food, Seafood, Restaurants, Cajun/Creole
3. Score: 2.6840 | Stars: 4.5 | American (Traditional), Chicken Shop, Southern, Re
4. Score: 2.6717 | Stars: 4.5 | Restaurants, American (Traditional), American (New
5. Score: 2.6264 | Stars: 4.5 | French, Restaurants, Cocktail Bars, Nightlife, Ame

In [ ]:
recs = get_top_k_recommendations(test_user, k=40)

Top 40 recommandations pour user qVc8ODYU5SZjKXVBgXdI7w:
1. Score: 2.9600 | Stars: 4.5 | Candy Stores, Shopping, Department Stores, Fast Fo
2. Score: 2.6908 | Stars: 4.0 | Live/Raw Food, Seafood, Restaurants, Cajun/Creole
3. Score: 2.6840 | Stars: 4.5 | American (Traditional), Chicken Shop, Southern, Re
4. Score: 2.6717 | Stars: 4.5 | Restaurants, American (Traditional), American (New
5. Score: 2.6264 | Stars: 4.5 | French, Restaurants, Cocktail Bars, Nightlife, Ame
6. Score: 2.5208 | Stars: 4.0 | German, Restaurants, Seafood, Cocktail Bars, Frenc
7. Score: 2.4920 | Stars: 4.5 | Nightlife, Bars, Food, Ethnic Food, Middle Eastern
8. Score: 2.4701 | Stars: 4.5 | Southern, American (New), Breakfast & Brunch, Rest
9. Score: 2.4405 | Stars: 4.5 | Restaurants, Breakfast & Brunch, Diners, Hawaiian,
10. Score: 2.4305 | Stars: 4.5 | Shopping, Butcher, Restaurants, Sandwiches, Food, 
11. Score: 2.4278 | Stars: 4.5 | Comfort Food, Restaurants, Southern
12. Score: 2.4190 | Stars: 4.0 | Burgers, Am

---

In [ ]:
def cold_start():
    pass